# Building Scene Reconstruction: Video → GLOMAP → Point Cloud → Mesh → Blender

This notebook takes a video walkaround/flythrough of a building and turns it into a textured
mesh you can import into Blender.

**Pipeline**

1. Extract frames from the video with `ffmpeg`
2. Extract SIFT features + match them with COLMAP
3. Run **GLOMAP** for fast global Structure-from-Motion (camera poses + sparse point cloud)
4. Densify with COLMAP's `patch_match_stereo` + `stereo_fusion` (dense point cloud)
5. Mesh the dense point cloud (Poisson reconstruction, via Open3D)
6. Clean + export the mesh as `.obj` / `.ply` for Blender

**Requirements** (install once, see next cell):
- `ffmpeg`
- [COLMAP](https://colmap.github.io/) (built with CUDA recommended for dense stereo, but CPU works)
- [GLOMAP](https://github.com/colmap/glomap) (binary `glomap` on PATH)
- Python: `pycolmap`, `open3d`, `numpy`, `trimesh`

> GLOMAP replaces COLMAP's incremental mapper with a much faster **global** SfM solver. It
> reads/writes COLMAP-format databases and sparse models, so everything downstream (dense
> stereo, meshing) still uses the normal COLMAP toolchain.


## 0. Config — set your paths here

In [ ]:
import os
from pathlib import Path

# ---- EDIT THESE ----
PROJECT_DIR = Path("./building_scan")      # everything will live under here
VIDEO_PATH  = Path("./input/building_walkthrough.mp4")  # your source video

FPS = 2                 # frames per second to extract (2-4 is a good start for SfM)
MAX_IMAGE_SIZE = 2048   # resize longest edge (0 = keep original)

CAMERA_MODEL = "OPENCV" # good default for consumer/phone/drone footage (has distortion params)
SINGLE_CAMERA = True    # set True if all frames came from the same camera/lens

# GPU flags (set to False if you don't have CUDA / a GPU-enabled COLMAP build)
USE_GPU_FEATURES = True
USE_GPU_DENSE = True
# ---------------------

IMAGES_DIR   = PROJECT_DIR / "images"
DB_PATH      = PROJECT_DIR / "database.db"
SPARSE_DIR   = PROJECT_DIR / "sparse"          # GLOMAP output (COLMAP sparse model format)
DENSE_DIR    = PROJECT_DIR / "dense"           # COLMAP dense workspace
MESH_DIR     = PROJECT_DIR / "mesh"

for d in [PROJECT_DIR, IMAGES_DIR, SPARSE_DIR, DENSE_DIR, MESH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())


## 1. Install dependencies

Run once. If you're on a machine that already has COLMAP/GLOMAP built, skip the apt/build
lines and just make sure `colmap` and `glomap` are on your `PATH`.


In [ ]:
# --- Python deps ---
%pip install pycolmap open3d numpy trimesh opencv-python tqdm -q


In [ ]:
# --- System deps (Ubuntu/Debian example) ---
# ffmpeg
!apt-get -qq update && apt-get -qq install -y ffmpeg

# COLMAP (CLI). On Ubuntu 22.04+/24.04 the apt package is usually recent enough for GLOMAP's
# database/model format. If you need a newer build, compile from source instead:
# https://colmap.github.io/install.html
!apt-get -qq install -y colmap

# GLOMAP does not ship as an apt package (as of writing) — build from source:
#   git clone https://github.com/colmap/glomap.git
#   cd glomap && mkdir build && cd build
#   cmake .. -GNinja && ninja && sudo ninja install
# After building, confirm it's discoverable:
!which glomap || echo "glomap not found on PATH — build it from https://github.com/colmap/glomap and re-run"


In [ ]:
# Sanity check versions
!ffmpeg -version | head -n 1
!colmap -h | head -n 1
!glomap -h | head -n 1


## 2. Extract frames from the video with ffmpeg

Uses a constant fps sample. For building walkarounds, 2-4 fps with decent camera motion
usually gives COLMAP/GLOMAP enough overlap between consecutive frames (aim for ~60-80%
visual overlap) without drowning the matcher in near-duplicate images.


In [ ]:
import subprocess

def extract_frames(video_path: Path, out_dir: Path, fps: int, max_size: int = 0):
    out_dir.mkdir(parents=True, exist_ok=True)
    # clear old frames so re-runs don't mix frame sets
    for f in out_dir.glob("*.jpg"):
        f.unlink()

    vf_filters = [f"fps={fps}"]
    if max_size and max_size > 0:
        # scale longest edge down to max_size, keep aspect ratio, only if larger
        vf_filters.append(
            f"scale='if(gt(iw,ih),min({max_size},iw),-2)':'if(gt(iw,ih),-2,min({max_size},ih))'"
        )
    vf = ",".join(vf_filters)

    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-vf", vf,
        "-qscale:v", "2",           # high JPEG quality (2 = near-lossless, lower is better)
        str(out_dir / "frame_%05d.jpg"),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    n = len(list(out_dir.glob("*.jpg")))
    print(f"Extracted {n} frames -> {out_dir}")
    return n

n_frames = extract_frames(VIDEO_PATH, IMAGES_DIR, FPS, MAX_IMAGE_SIZE)
assert n_frames > 0, "No frames extracted — check VIDEO_PATH"


### Optional: blur filtering

Motion-blurred frames hurt feature matching. This drops the blurriest frames using a
Laplacian-variance sharpness score.


In [ ]:
import cv2
import numpy as np

def sharpness_score(img_path: Path) -> float:
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    return cv2.Laplacian(img, cv2.CV_64F).var()

def filter_blurry_frames(img_dir: Path, keep_fraction: float = 0.9):
    frames = sorted(img_dir.glob("*.jpg"))
    scores = [(f, sharpness_score(f)) for f in frames]
    scores.sort(key=lambda x: x[1])  # ascending: blurriest first
    n_drop = int(len(scores) * (1 - keep_fraction))
    dropped = scores[:n_drop]
    for f, s in dropped:
        f.unlink()
    print(f"Dropped {len(dropped)} blurry frames, kept {len(frames) - len(dropped)}")

# Uncomment to enable (drops the blurriest 10% of frames):
# filter_blurry_frames(IMAGES_DIR, keep_fraction=0.9)


## 3. COLMAP: feature extraction + matching

GLOMAP consumes a COLMAP database, so we still use `colmap feature_extractor` and a matcher
to populate it. `sequential_matcher` is a good fit for video frames (it matches each frame to
its temporal neighbors, plus optional loop-closure checks), and is much faster than exhaustive
matching for long walkthroughs.


In [ ]:
def run(cmd):
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# --- Feature extraction ---
run([
    "colmap", "feature_extractor",
    "--database_path", str(DB_PATH),
    "--image_path", str(IMAGES_DIR),
    "--ImageReader.camera_model", CAMERA_MODEL,
    "--ImageReader.single_camera", "1" if SINGLE_CAMERA else "0",
    "--SiftExtraction.use_gpu", "1" if USE_GPU_FEATURES else "0",
])


In [ ]:
# --- Feature matching ---
# sequential_matcher: good default for video (temporal order in filenames matters here,
# which is why ffmpeg's zero-padded frame_%05d.jpg naming above is important).
run([
    "colmap", "sequential_matcher",
    "--database_path", str(DB_PATH),
    "--SiftMatching.use_gpu", "1" if USE_GPU_FEATURES else "0",
    "--SequentialMatching.loop_detection", "1",   # helps close the loop on a walkaround
    "--SequentialMatching.overlap", "10",
])

# If your video pans around and revisits the same facade non-sequentially (e.g. you looped
# the building more than once, or clipped/reordered footage), exhaustive matching is more
# robust at the cost of speed:
# run([
#     "colmap", "exhaustive_matcher",
#     "--database_path", str(DB_PATH),
#     "--SiftMatching.use_gpu", "1" if USE_GPU_FEATURES else "0",
# ])


## 4. GLOMAP: global structure-from-motion

This is the step that replaces COLMAP's (slower, incremental) `mapper` with GLOMAP's global
pose-graph solver. Output is a standard COLMAP sparse reconstruction (`cameras.bin`,
`images.bin`, `points3D.bin`) under `SPARSE_DIR/0`.


In [ ]:
run([
    "glomap", "mapper",
    "--database_path", str(DB_PATH),
    "--image_path", str(IMAGES_DIR),
    "--output_path", str(SPARSE_DIR),
])

# GLOMAP writes one or more reconstructions (like COLMAP mapper does when the scene breaks
# into disconnected components). List what we got:
sparse_models = sorted([d for d in SPARSE_DIR.iterdir() if d.is_dir()])
print("Sparse models:", sparse_models)
assert sparse_models, "GLOMAP produced no sparse model — check the matching step / image overlap"

# Use the largest reconstruction (most registered images) if there's more than one.
def n_registered_images(model_dir: Path) -> int:
    import pycolmap
    rec = pycolmap.Reconstruction(str(model_dir))
    return len(rec.images)

best_model = max(sparse_models, key=n_registered_images)
print("Using model:", best_model, "with", n_registered_images(best_model), "registered images")


### Quick look at the sparse point cloud

In [ ]:
import pycolmap

rec = pycolmap.Reconstruction(str(best_model))
print(rec.summary())

sparse_ply = PROJECT_DIR / "sparse_points.ply"
rec.export_PLY(str(sparse_ply))
print("Exported sparse point cloud ->", sparse_ply)


## 5. Dense reconstruction (COLMAP MVS)

The sparse point cloud from SfM is too sparse to mesh well. We densify it with COLMAP's
multi-view stereo (`patch_match_stereo` + `stereo_fusion`) using the poses GLOMAP just solved.

This step is the most compute-heavy part of the pipeline — expect it to take a while for
a few hundred frames, especially on CPU.


In [ ]:
dense_workspace = DENSE_DIR

run([
    "colmap", "image_undistorter",
    "--image_path", str(IMAGES_DIR),
    "--input_path", str(best_model),
    "--output_path", str(dense_workspace),
    "--output_type", "COLMAP",
])


In [ ]:
run([
    "colmap", "patch_match_stereo",
    "--workspace_path", str(dense_workspace),
    "--workspace_format", "COLMAP",
    "--PatchMatchStereo.geom_consistency", "true",
    "--PatchMatchStereo.gpu_index", "0" if USE_GPU_DENSE else "-1",
])


In [ ]:
dense_ply = dense_workspace / "fused.ply"

run([
    "colmap", "stereo_fusion",
    "--workspace_path", str(dense_workspace),
    "--workspace_format", "COLMAP",
    "--input_type", "geometric",
    "--output_path", str(dense_ply),
])

print("Dense point cloud ->", dense_ply)


## 6. Mesh the dense point cloud (Open3D)

Poisson surface reconstruction gives a closed, watertight-ish mesh — good for buildings.
We also crop the mesh back to the point cloud's bounding box, since Poisson tends to
extrapolate/balloon past the actual data in low-density regions.


In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.io.read_point_cloud(str(dense_ply))
print(pcd)

# --- Clean up ---
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# Estimate normals (required for Poisson reconstruction)
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(k=30)

o3d.io.write_point_cloud(str(MESH_DIR / "cleaned_points.ply"), pcd)


In [ ]:
# --- Poisson meshing ---
POISSON_DEPTH = 10   # higher = more detail + more memory/time (9-12 typical range)

mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=POISSON_DEPTH
)

# Trim low-density (extrapolated / unsupported) regions
densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.02)  # drop bottom 2% density vertices
vertices_to_remove = densities < density_threshold
mesh.remove_vertices_by_mask(vertices_to_remove)

print(mesh)


In [ ]:
# --- Crop mesh to the point cloud's bounding box (removes Poisson's outward "bubbles") ---
bbox = pcd.get_axis_aligned_bounding_box()
mesh = mesh.crop(bbox)

# --- Clean topology ---
mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles()
mesh.remove_degenerate_triangles()
mesh.remove_unreferenced_vertices()
mesh.compute_vertex_normals()

print("Final mesh:", mesh)


### Optional: transfer color from the point cloud to the mesh

Poisson reconstruction can carry input colors through automatically if the input point cloud
has them (COLMAP's fused.ply does). If your mesh came out uncolored for any reason, this cell
nearest-neighbor-samples color from the point cloud instead.


In [ ]:
if not mesh.has_vertex_colors():
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    colors = np.zeros((len(mesh.vertices), 3))
    pcd_colors = np.asarray(pcd.colors)
    for i, v in enumerate(np.asarray(mesh.vertices)):
        _, idx, _ = pcd_tree.search_knn_vector_3d(v, 1)
        colors[i] = pcd_colors[idx[0]]
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    print("Transferred vertex colors from point cloud")
else:
    print("Mesh already has vertex colors")


## 7. Export for Blender

`.obj` is the safest bet for full compatibility (Blender's importer handles OBJ + vertex
colors/materials reliably). We also keep a `.ply` since it round-trips vertex colors more
simply if you don't need UVs/materials.


In [ ]:
obj_path = MESH_DIR / "building_mesh.obj"
ply_path = MESH_DIR / "building_mesh.ply"

o3d.io.write_triangle_mesh(str(obj_path), mesh, write_vertex_colors=True)
o3d.io.write_triangle_mesh(str(ply_path), mesh, write_vertex_colors=True)

print("Exported:")
print(" -", obj_path.resolve())
print(" -", ply_path.resolve())


### Importing into Blender

- `File > Import > Wavefront (.obj)` (or `.ply`), select the file above.
- If colors look flat/gray: in the Shading tab, add an **Attribute** node (name `Col`) or
  **Vertex Color** node feeding into your Principled BSDF's Base Color — OBJ/PLY vertex
  colors aren't auto-wired into materials by Blender's importer.
- Scale: COLMAP/GLOMAP reconstructions are in an arbitrary unit scale (not meters) unless you
  used ground control points / known distances. If you need real-world scale, measure a known
  distance in the scene (e.g. a floor-to-floor height) and apply a uniform scale in Blender to
  match.
- The reconstruction's up-axis may not match Blender's Z-up convention — you may need to
  rotate the imported object (commonly -90° on X) depending on how COLMAP's camera-relative
  coordinate frame landed.


## 8. (Optional) Visualize point cloud / mesh inline

In [ ]:
# Quick non-interactive preview render (headless-friendly). For an interactive window,
# use o3d.visualization.draw_geometries([mesh]) instead when running locally with a display.

vis = o3d.visualization.Visualizer()
vis.create_window(visible=False)
vis.add_geometry(mesh)
vis.poll_events()
vis.update_renderer()
vis.capture_screen_image(str(MESH_DIR / "mesh_preview.png"))
vis.destroy_window()

from IPython.display import Image
Image(filename=str(MESH_DIR / "mesh_preview.png"))


## Notes / troubleshooting

- **GLOMAP registers few/no images**: usually a matching problem, not a GLOMAP problem —
  increase `--SequentialMatching.overlap`, lower `FPS` isn't the fix (more overlap between
  frames helps more than more frames), or switch to `exhaustive_matcher` for smaller datasets.
- **Dense reconstruction is very slow**: reduce `MAX_IMAGE_SIZE`, or set
  `PatchMatchStereo.geom_consistency` to `false` for a rougher/faster pass first.
- **Mesh has holes on featureless walls**: textureless facades (flat stucco, glass) give SfM
  little to match on. Consider capturing footage with more oblique angles / texture context,
  or increasing `POISSON_DEPTH` won't fix missing geometry — you need more source views of
  that surface.
- **Alternative to Poisson**: for scenes with very uneven point density, try
  `o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting` instead — it tends to
  hallucinate less but leaves more holes.
